In [2]:
import duckdb
import pandas as pd

Create Development Database

In [ ]:
# First create data folder in the project before creating duckdb below: mkdir data
sql_query = '''
show tables
'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
    display(con.sql(sql_query).df())

,name


Create Prodction Database

In [ ]:
# First create data folder in the project before creating duckdb below: mkdir data
sql_query = '''
show tables
'''

with duckdb.connect('data/prd_crm_erp_database.db') as con:
    display(con.sql(sql_query).df())

,name


Import CSV to the Database

In [ ]:
sql_query_import_1 = '''
CREATE OR REPLACE TABLE raw_erp_cust_az12 AS 
SELECT *
FROM read_csv_auto('data/cust_az12.csv', normalize_names=True)
''' 

sql_query_import_2 = '''
CREATE OR REPLACE TABLE raw_erp_loc_a101 AS 
SELECT *
FROM read_csv_auto('data/loc_a101.csv', normalize_names=True)
''' 

sql_query_import_3 = '''
CREATE OR REPLACE TABLE raw_erp_px_cat_g1v2 AS 
SELECT *
FROM read_csv_auto('data/px_cat_g1v2.csv', normalize_names=True)
''' 

sql_query_import_4 = '''
CREATE OR REPLACE TABLE raw_crm_cust_info AS 
SELECT *
FROM read_csv_auto('data/cust_info.csv', normalize_names=True)
''' 

sql_query_import_5 = '''
CREATE OR REPLACE TABLE raw_crm_prd_info AS 
SELECT *
FROM read_csv_auto('data/prd_info.csv', normalize_names=True)
''' 

sql_query_import_6 = '''
CREATE OR REPLACE TABLE raw_crm_sales_details AS 
SELECT *
FROM read_csv_auto('data/sales_details.csv', normalize_names=True)
''' 

with duckdb.connect('data/dev_crm_erp_database.db') as con:
    con.sql(sql_query_import_1)
    con.sql(sql_query_import_2)
    con.sql(sql_query_import_3)
    con.sql(sql_query_import_4)
    con.sql(sql_query_import_5)
    con.sql(sql_query_import_6)

Check the Impoarted Table

In [ ]:
# First create data folder in the project before creating duckdb below: mkdir data
sql_query = '''
show tables
'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
    display(con.sql(sql_query).df())

,name
0,bronze.crm_cust_info
1,bronze.crm_prd_info
2,bronze.crm_sales_details
3,bronze.erp_cust_az12
4,bronze.erp_loc_a101
5,bronze.erp_px_cat_g1v2
6,bronze_crm_cust_info
7,bronze_crm_prd_info
8,bronze_crm_sales_details
9,bronze_erp_cust_az12


### Import CSV to prod

In [60]:
sql_query_import_1 = '''
CREATE OR REPLACE TABLE raw_erp_cust_az12 AS 
SELECT *
FROM read_csv_auto('data/cust_az12.csv', normalize_names=True)
''' 

sql_query_import_2 = '''
CREATE OR REPLACE TABLE raw_erp_loc_a101 AS 
SELECT *
FROM read_csv_auto('data/loc_a101.csv', normalize_names=True)
''' 

sql_query_import_3 = '''
CREATE OR REPLACE TABLE raw_erp_px_cat_g1v2 AS 
SELECT *
FROM read_csv_auto('data/px_cat_g1v2.csv', normalize_names=True)
''' 

sql_query_import_4 = '''
CREATE OR REPLACE TABLE raw_crm_cust_info AS 
SELECT *
FROM read_csv_auto('data/cust_info.csv', normalize_names=True)
''' 

sql_query_import_5 = '''
CREATE OR REPLACE TABLE raw_crm_prd_info AS 
SELECT *
FROM read_csv_auto('data/prd_info.csv', normalize_names=True)
''' 

sql_query_import_6 = '''
CREATE OR REPLACE TABLE raw_crm_sales_details AS 
SELECT *
FROM read_csv_auto('data/sales_details.csv', normalize_names=True)
''' 

with duckdb.connect('data/prd_crm_erp_database.db') as con:
    con.sql(sql_query_import_1)
    con.sql(sql_query_import_2)
    con.sql(sql_query_import_3)
    con.sql(sql_query_import_4)
    con.sql(sql_query_import_5)
    con.sql(sql_query_import_6)

### Check inmport tables

In [61]:
# First create data folder in the project before creating duckdb below: mkdir data
sql_query = '''
show tables
'''

with duckdb.connect('data/prd_crm_erp_database.db') as con:
    display(con.sql(sql_query).df())

,name
0,raw_crm_cust_info
1,raw_crm_prd_info
2,raw_crm_sales_details
3,raw_erp_cust_az12
4,raw_erp_loc_a101
5,raw_erp_px_cat_g1v2


## Data Validation

### 1. CRM Data

In [33]:
sql_query = '''

SELECT * 
FROM bronze_crm_cust_info LIMIT 5

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df())     

,cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
0,11000,AW00011000,Jon,Yang,M,M,2025-10-06
1,11001,AW00011001,Eugene,Huang,S,M,2025-10-06
2,11002,AW00011002,Ruben,Torres,M,M,2025-10-06
3,11003,AW00011003,Christy,Zhu,S,F,2025-10-06
4,11004,AW00011004,Elizabeth,Johnson,S,F,2025-10-06


In [26]:
sql_query = '''

SELECT cst_id, count(*) 
FROM bronze_crm_cust_info
GROUP BY cst_id
HAVING COUNT(*) > 1

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df())

,cst_id,count_star()
0,NaN,4
1,29449.0,2
2,29433.0,2
3,29466.0,3
4,29473.0,2
5,29483.0,2


In [49]:
sql_query = '''

SELECT cst_id, count(*) 



FROM ( 

    SELECT *
    , ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) AS rn 
    FROM bronze_crm_cust_info 
    WHERE cst_id IS NOT NULL 

)

WHERE rn = 1
GROUP BY cst_id
HAVING COUNT(*) > 1


'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df())

,cst_id,count_star()


### Check Empty Spaces

In [14]:
sql_query = '''

SELECT * 
FROM bronze_crm_cust_info 
WHERE cst_firstname != TRIM(cst_firstname)

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df())   

,cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
0,11000,AW00011000,Jon,Yang,M,M,2025-10-06
1,11004,AW00011004,Elizabeth,Johnson,S,F,2025-10-06
2,11012,AW00011012,Lauren,Walker,M,F,2025-10-06
3,11013,AW00011013,Ian,Jenkins,M,M,2025-10-06
4,11015,AW00011015,Chloe,Young,S,F,2025-10-06
5,11021,AW00011021,Destiny,Wilson,S,F,2025-10-07
6,11063,AW00011063,Angela,Murphy,S,F,2025-10-07
7,11065,AW00011065,Jessica,Henderson,M,F,2025-10-07
8,11067,AW00011067,Caleb,Carter,S,M,2025-10-07
9,11070,AW00011070,Willie,Raji,M,M,2025-10-07


### Data Standardization

In [2]:
sql_query = '''

SELECT DISTINCT cst_gndr-- cst_marital_status
FROM bronze_crm_cust_info 
-- WHERE cst_firstname != TRIM(cst_firstname)

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df())   

,cst_gndr
0,F
1,None
2,M


### Data Cleansing

In [ ]:
sql_query = ''' 

SELECT
 cst_id, 
    cst_key, 
    TRIM(cst_firstname) AS cst_firstname,
    TRIM(cst_lastname) AS cst_lastname,
    CASE UPPER(TRIM(cst_marital_status))
        WHEN 'M' THEN 'Married'
        WHEN 'S' THEN 'Single'
        ELSE 'n/a'
    END AS cst_material_status, 
    CASE UPPER(TRIM(cst_gndr))
        WHEN 'M' THEN 'Male'
        WHEN 'F' THEN 'Female'
        ELSE 'n/a'
    END AS cst_gndr,
    cst_create_date

FROM ( 

    SELECT *
    , ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) AS rn 
    FROM bronze_crm_cust_info 
    WHERE cst_id IS NOT NULL 

)

WHERE rn = 1
    
'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df())

ParserException: Parser Error: syntax error at or near "{"

### Bronze Layer: Column Casting Process

In [20]:
sql_query = '''

WITH source AS (

    SELECT * 
    FROM raw_crm_prd_info

) 

SELECT 
    CAST(prd_id AS INT) AS prd_id,
    CAST(prd_key AS NVARCHAR(50)) AS prd_key,
    CAST(prd_nm AS NVARCHAR(50)) AS prd_nm, 
    CAST(prd_cost AS INT) AS prd_cost,
    CAST(prd_line AS NVARCHAR(50)) AS prd_line,
    CAST(prd_start_dt AS DATE) AS prd_start_dt,
    CAST(prd_end_dt AS DATE) AS prd_end_dt

FROM source
LIMIT 10

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df()) 

,prd_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
0,210,CO-RF-FR-R92B-58,HL Road Frame - Black- 58,NaN,R,2003-07-01,NaT
1,211,CO-RF-FR-R92R-58,HL Road Frame - Red- 58,NaN,R,2003-07-01,NaT
2,212,AC-HE-HL-U509-R,Sport-100 Helmet- Red,12.0,S,2011-07-01,2007-12-28
3,213,AC-HE-HL-U509-R,Sport-100 Helmet- Red,14.0,S,2012-07-01,2008-12-27
4,214,AC-HE-HL-U509-R,Sport-100 Helmet- Red,13.0,S,2013-07-01,NaT
5,215,AC-HE-HL-U509,Sport-100 Helmet- Black,12.0,S,2011-07-01,2007-12-28
6,216,AC-HE-HL-U509,Sport-100 Helmet- Black,14.0,S,2012-07-01,2008-12-27
7,217,AC-HE-HL-U509,Sport-100 Helmet- Black,13.0,S,2013-07-01,NaT
8,218,CL-SO-SO-B909-M,Mountain Bike Socks- M,3.0,M,2011-07-01,2007-12-28
9,219,CL-SO-SO-B909-L,Mountain Bike Socks- L,3.0,M,2011-07-01,2007-12-28


In [11]:
sql_query = '''

WITH source AS (

    SELECT * 
    FROM raw_crm_sales_details

) 

SELECT 
    CAST(sls_ord_num AS NVARCHAR(50)) AS sls_ord_num,
    CAST(sls_prd_key AS NVARCHAR(50)) AS sls_prd_key,
    CAST(sls_cust_id AS NVARCHAR(50)) AS sls_cust_id,
    STRPTIME(CAST(sls_order_dt AS VARCHAR), '%Y%m%d') AS sls_order_dt,
    STRPTIME(CAST(sls_ship_dt AS VARCHAR), '%Y%m%d') AS sls_ship_dt,
    STRPTIME(CAST(sls_due_dt AS VARCHAR), '%Y%m%d') AS sls_due_dt,
    CAST(sls_sales AS DECIMAL(20,2)) AS sls_sales, 
    CAST(sls_sales AS INT) AS sls_quantity, 
    CAST(sls_price AS DECIMAL(20,2)) AS sls_price

FROM source
LIMIT 10

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df()) 

,sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
0,SO43697,BK-R93R-62,21768,2010-12-29,2011-01-05,2011-01-10,3578.0,3578,3578.0
1,SO43698,BK-M82S-44,28389,2010-12-29,2011-01-05,2011-01-10,3400.0,3400,3400.0
2,SO43699,BK-M82S-44,25863,2010-12-29,2011-01-05,2011-01-10,3400.0,3400,3400.0
3,SO43700,BK-R50B-62,14501,2010-12-29,2011-01-05,2011-01-10,699.0,699,699.0
4,SO43701,BK-M82S-44,11003,2010-12-29,2011-01-05,2011-01-10,3400.0,3400,3400.0
5,SO43702,BK-R93R-44,27645,2010-12-30,2011-01-06,2011-01-11,3578.0,3578,3578.0
6,SO43703,BK-R93R-62,16624,2010-12-30,2011-01-06,2011-01-11,3578.0,3578,3578.0
7,SO43704,BK-M82B-48,11005,2010-12-30,2011-01-06,2011-01-11,3375.0,3375,3375.0
8,SO43705,BK-M82S-38,11011,2010-12-30,2011-01-06,2011-01-11,3400.0,3400,3400.0
9,SO43706,BK-R93R-48,27621,2010-12-31,2011-01-07,2011-01-12,3578.0,3578,3578.0


### DATA Cleansing: prd_info

In [ ]:
sql_query = '''

WITH products AS (

    SELECT * 
    FROM bronze_crm_prd_info

) 

SELECT
    prd_id,
    TRIM(REPLACE(SUBSTRING(prd_key, 1, 5), '-', '_')) AS cat_id, 
	TRIM(SUBSTRING(prd_key, 7, LEN(prd_key))) AS prd_key,  
	prd_nm, 
	CASE WHEN prd_cost IS NULL THEN 0 ELSE prd_cost END AS prd_cost, 
	CASE UPPER(TRIM(prd_line))
		WHEN 'S' then 'Other Sales'
		WHEN 'T' then 'Touring'
		WHEN 'M' then 'Mountain'
		WHEN 'R' then 'Road'
		ELSE 'Unknown' 
	END AS prd_line,
	CAST(prd_start_dt AS date) AS prd_start_dt, 
	-- CAST(DATEADD(day, -1, lead(prd_start_dt) OVER (PARTITION BY prd_key ORDER BY prd_start_dt)) as date) as prd_end_dt 
    lead(prd_start_dt) OVER (PARTITION BY prd_key ORDER BY prd_start_dt) - 1 as prd_end_dt 

FROM products

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df()) 

,prd_id,cat_id,prd_key,prd_nm,prd_cost,prd_line,prd_start_dt,prd_end_dt
0,483,AC_BR,RA-H123,Hitch Rack - 4-Bike,45,Other Sales,2013-07-01,NaT
1,599,BI_MB,BK-M18B-48,Mountain-500 Black- 48,295,Mountain,2013-07-01,NaT
2,360,BI_MB,BK-M68B-42,Mountain-200 Black- 42,1106,Mountain,2012-07-01,2013-06-30
3,361,BI_MB,BK-M68B-42,Mountain-200 Black- 42,1252,Mountain,2013-07-01,NaT
4,370,BI_RB,BK-R89R-52,Road-250 Red- 52,1519,Road,2012-07-01,NaT
...,...,...,...,...,...,...,...,...
392,258,CO_RF,FR-R38B-60,LL Road Frame - Black- 60,205,Road,2013-07-01,NaT
393,262,CO_RF,FR-R38R-44,LL Road Frame - Red- 44,181,Road,2011-07-01,2012-06-30
394,263,CO_RF,FR-R38R-44,LL Road Frame - Red- 44,187,Road,2012-07-01,NaT
395,423,CO_WH,RW-R762,ML Road Rear Wheel,122,Road,2012-07-01,NaT


### Data Cleansing: Sales_Details

In [39]:
sql_query = '''

WITH sales AS (

    SELECT * 
    FROM bronze_crm_sales_details

) 

select 
	sls_ord_num, 
	sls_prd_key, 
	sls_cust_id,
	CAST(STRPTIME(CAST(CASE 
		WHEN sls_order_dt <= 0 OR LEN(CAST(sls_order_dt AS STRING)) != 8 THEN NULL 
		ELSE sls_order_dt 
    END AS VARCHAR), '%Y%M%d') AS DATE) AS sls_order_dt, 
	CAST(STRPTIME(CAST(sls_ship_dt AS VARCHAR), '%Y%M%d') AS DATE) AS sls_ship_dt, 
	CAST(STRPTIME(CAST(sls_due_dt AS VARCHAR), '%Y%M%d') AS DATE) AS sls_due_dt, 
    CASE 
		WHEN sls_sales IS NULL OR sls_sales <= 0 OR sls_sales != sls_quantity * ABS(sls_price) THEN sls_quantity * ABS(sls_price) 
		ELSE sls_sales 
    END as sls_sales, 
	sls_quantity, 
	CASE 
		WHEN sls_price IS NULL OR sls_price <= 0
		THEN sls_sales / NULLIF(sls_quantity, 0)
		ELSE sls_price
    END AS sls_price

FROM sales

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df()) 

,sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price
0,SO43697,BK-R93R-62,21768,2010-01-29,2011-01-05,2011-01-10,12802084.0,3578.0,3578.0
1,SO43698,BK-M82S-44,28389,2010-01-29,2011-01-05,2011-01-10,11560000.0,3400.0,3400.0
2,SO43699,BK-M82S-44,25863,2010-01-29,2011-01-05,2011-01-10,11560000.0,3400.0,3400.0
3,SO43700,BK-R50B-62,14501,2010-01-29,2011-01-05,2011-01-10,488601.0,699.0,699.0
4,SO43701,BK-M82S-44,11003,2010-01-29,2011-01-05,2011-01-10,11560000.0,3400.0,3400.0
...,...,...,...,...,...,...,...,...,...
60393,SO75122,FE-6654,15868,2014-01-28,2014-01-04,2014-01-09,484.0,22.0,22.0
60394,SO75122,CA-1098,15868,2014-01-28,2014-01-04,2014-01-09,81.0,9.0,9.0
60395,SO75123,FE-6654,18759,2014-01-28,2014-01-04,2014-01-09,484.0,22.0,22.0
60396,SO75123,ST-1401,18759,2014-01-28,2014-01-04,2014-01-09,25281.0,159.0,159.0


In [41]:
sql_query = '''

WITH customers AS (

    SELECT *
    FROM bronze_erp_cust_az12 

)

SELECT 
    CASE 
        WHEN cid LIKE 'NAS%' THEN SUBSTRING(TRIM(cid), 4, LEN(TRIM(cid)))
		ELSE TRIM(cid) 
    END AS cid, 
	CASE 
        WHEN bdate > CURRENT_DATE() THEN NULL 
        ELSE bdate 
    END AS bdate, 
	CASE  
		WHEN UPPER(TRIM(gen)) IS NULL OR UPPER(TRIM(gen)) = ''  THEN 'Unknown'
		WHEN UPPER(TRIM(gen)) = 'M' then 'Male'
		WHEN UPPER(TRIM(gen)) = 'F' then 'Female' 
		ELSE TRIM(gen) 
	END AS gen

FROM customers

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df()) 

,cid,bdate,gen
0,AW00011000,1971-10-06,Male
1,AW00011001,1976-05-10,Male
2,AW00011002,1971-02-09,Male
3,AW00011003,1973-08-14,Female
4,AW00011004,1979-08-05,Female
...,...,...,...
18479,AW00029479,1969-06-30,Unknown
18480,AW00029480,1977-05-06,Unknown
18481,AW00029481,1965-07-04,Unknown
18482,AW00029482,1964-09-01,Unknown


In [44]:
sql_query = '''

WITH locations AS (

    SELECT * FROM bronze_erp_loc_a101

)

SELECT 

	REPLACE(TRIM(cid),'-', '') AS cid, 
	CASE 
		WHEN UPPER(TRIM(cntry)) IS NULL OR UPPER(TRIM(cntry)) = '' THEN 'n/a'
		when UPPER(TRIM(cntry)) IN ('USA', 'US') THEN 'United States' 
		when UPPER(TRIM(cntry)) = 'DE' THEN 'Germany'
		ELSE TRIM(cntry)
	END cntry

FROM locations 

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df()) 

,cid,cntry
0,AW00011000,Australia
1,AW00011001,Australia
2,AW00011002,Australia
3,AW00011003,Australia
4,AW00011004,Australia
...,...,...
18479,AW00029479,France
18480,AW00029480,United Kingdom
18481,AW00029481,Germany
18482,AW00029482,France


In [48]:
sql_query = '''

WITH category AS (

    SELECT * FROM bronze_erp_px_cat_g1v2

) 

SELECT 
	id,
	cat,
	subcat, 
	maintenance 
		
FROM category 

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df())

,ID,CAT,SUBCAT,MAINTENANCE
0,AC_BR,Accessories,Bike Racks,true
1,AC_BS,Accessories,Bike Stands,false
2,AC_BC,Accessories,Bottles and Cages,false
3,AC_CL,Accessories,Cleaners,true
4,AC_FE,Accessories,Fenders,false
5,AC_HE,Accessories,Helmets,true
6,AC_HP,Accessories,Hydration Packs,false
7,AC_LI,Accessories,Lights,true
8,AC_LO,Accessories,Locks,true
9,AC_PA,Accessories,Panniers,false


In [54]:
sql_query = '''

WITH source AS (

    SELECT * 
    FROM raw_crm_cust_info 

) 

SELECT 
    CAST(cst_id AS INT) AS cst_id,
    CAST(cst_key AS NVARCHAR(50)) AS cst_key,
    CAST(cst_firstname AS NVARCHAR(50)) AS cst_firstname,
    CAST(cst_lastname AS NVARCHAR(50)) AS cst_lastname,
    CAST(cst_marital_status AS NVARCHAR(50)) AS cst_marital_status,
    CAST(cst_gndr AS NVARCHAR(50)) AS cst_gndr, 
    CAST(cst_create_date AS DATE) AS cst_create_date, 
    NOW() AS dwh_create_date

FROM source

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df())

,cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date,dwh_create_date
0,11000.0,AW00011000,Jon,Yang,M,M,2025-10-06,2025-03-20 03:49:33.133000+00:00
1,11001.0,AW00011001,Eugene,Huang,S,M,2025-10-06,2025-03-20 03:49:33.133000+00:00
2,11002.0,AW00011002,Ruben,Torres,M,M,2025-10-06,2025-03-20 03:49:33.133000+00:00
3,11003.0,AW00011003,Christy,Zhu,S,F,2025-10-06,2025-03-20 03:49:33.133000+00:00
4,11004.0,AW00011004,Elizabeth,Johnson,S,F,2025-10-06,2025-03-20 03:49:33.133000+00:00
...,...,...,...,...,...,...,...,...
18489,29482.0,AW00029482,Clayton,Zhang,M,None,2026-01-25,2025-03-20 03:49:33.133000+00:00
18490,29483.0,AW00029483,None,Navarro,None,None,2026-01-25,2025-03-20 03:49:33.133000+00:00
18491,29483.0,AW00029483,Marc,Navarro,M,None,2026-01-27,2025-03-20 03:49:33.133000+00:00
18492,NaN,13451235,None,None,None,None,NaT,2025-03-20 03:49:33.133000+00:00


In [ ]:
sql_query = '''

    SELECT * 
    FROM gold_sales_fact
    WHERE sales_amount != quantity * price 

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df())

,order_number,product_key,customer_key,order_date,shipping_date,due_date,sales_amount,quantity,price
0,SO57804,277,5471,2013-01-11,2013-01-18,2013-01-23,591361.0,769,1.0
1,SO57916,225,12025,2013-01-13,2013-01-20,2013-01-25,900.0,30,1.0
2,SO58326,174,1046,2013-01-20,2013-01-27,2013-01-01,484.0,22,1.0
3,SO58623,271,6013,2013-01-25,2013-01-01,2013-01-06,2893401.0,1701,1.0
4,SO58818,227,10388,2013-01-28,2013-01-04,2013-01-09,441.0,21,1.0


In [59]:
sql_query = '''

SELECT * 
FROM gold_sales_fact --  bronze_crm_sales_details-- gold_sales_fact 
-- WHERE order_date > shipping_date -- or shipping_date > due_date or order_date > due_date

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df())

,order_number,product_key,customer_key,order_date,shipping_date,due_date,sales_amount,quantity,price
0,SO43700,46,3502,2010-01-29,2011-01-05,2011-01-10,488601.0,699.0,699.0
1,SO43702,15,16646,2010-01-30,2011-01-06,2011-01-11,12802084.0,3578.0,3578.0
2,SO43704,31,6,2010-01-30,2011-01-06,2011-01-11,11390625.0,3375.0,3375.0
3,SO43705,24,12,2010-01-30,2011-01-06,2011-01-11,11560000.0,3400.0,3400.0
4,SO43706,16,16622,2010-01-31,2011-01-07,2011-01-12,12802084.0,3578.0,3578.0
...,...,...,...,...,...,...,...,...,...
60393,SO75024,157,11821,2014-01-25,2014-01-01,2014-01-06,576.0,24.0,24.0
60394,SO75033,101,5176,2014-01-26,2014-01-02,2014-01-07,1225.0,35.0,35.0
60395,SO75087,101,795,2014-01-28,2014-01-04,2014-01-09,1225.0,35.0,35.0
60396,SO75105,101,4161,2014-01-28,2014-01-04,2014-01-09,1225.0,35.0,35.0


In [3]:
sql_query = '''

select * from "dev_crm_erp_database"."main_dbt_test__audit"."source_not_null_dev_crm_erp_database_raw_crm_cust_info_cst_id"

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df())

,cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
0,NaN,SF566,None,None,None,None,NaT
1,NaN,PO25,None,None,None,None,NaT
2,NaN,13451235,None,None,None,None,NaT
3,NaN,A01Ass,None,None,None,None,NaT


In [12]:
sql_query = '''

  select * from "dev_crm_erp_database"."main_dbt_test__audit"."sales_amount_cals"

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df())

,order_number,product_key,customer_key,order_date,shipping_date,due_date,sales_amount,quantity,price
0,SO57804,277,5471,2013-01-11,2013-01-18,2013-01-23,591361.0,769,1.0
1,SO57916,225,12025,2013-01-13,2013-01-20,2013-01-25,900.0,30,1.0
2,SO58326,174,1046,2013-01-20,2013-01-27,2013-01-01,484.0,22,1.0
3,SO58623,271,6013,2013-01-25,2013-01-01,2013-01-06,2893401.0,1701,1.0
4,SO58818,227,10388,2013-01-28,2013-01-04,2013-01-09,441.0,21,1.0


In [39]:
sql_query = '''
 
 select * from silver_crm_sales_details where sls_ord_num = 'SO57916'
 -- select * from gold_sales_fact where order_number = 'SO57916'

 

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df())

,sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price,dwh_create_date
0,SO57916,TT-M928,23024,2013-01-13,2013-01-20,2013-01-25,25.0,5,5.0,2025-04-04 20:48:39.513000+00:00
1,SO57916,TI-M602,23024,2013-01-13,2013-01-20,2013-01-25,900.0,30,1.0,2025-04-04 20:48:39.513000+00:00
2,SO57916,FE-6654,23024,2013-01-13,2013-01-20,2013-01-25,484.0,22,22.0,2025-04-04 20:48:39.513000+00:00
3,SO57916,LJ-0192-X,23024,2013-01-13,2013-01-20,2013-01-25,2500.0,50,50.0,2025-04-04 20:48:39.513000+00:00


In [43]:
ql_query = '''

select * from "dev_crm_erp_database"."main_dbt_test__audit"."orders_of_dates"

 

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df())

,sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price,dwh_create_date
0,SO57916,TT-M928,23024,2013-01-13,2013-01-20,2013-01-25,25.0,5,5.0,2025-04-04 20:51:41.573000+00:00
1,SO57916,TI-M602,23024,2013-01-13,2013-01-20,2013-01-25,900.0,30,1.0,2025-04-04 20:51:41.573000+00:00
2,SO57916,FE-6654,23024,2013-01-13,2013-01-20,2013-01-25,484.0,22,22.0,2025-04-04 20:51:41.573000+00:00
3,SO57916,LJ-0192-X,23024,2013-01-13,2013-01-20,2013-01-25,2500.0,50,50.0,2025-04-04 20:51:41.573000+00:00


In [49]:
ql_query = '''

SELECT * 
FROM silver_crm_sales_details -- gold_sales_fact 
-- WHERE order_date < shipping_date -- or shipping_date < due_date or order_date < due_date

 

'''

with duckdb.connect('data/dev_crm_erp_database.db') as con:
       display(con.sql(sql_query).df())

,sls_ord_num,sls_prd_key,sls_cust_id,sls_order_dt,sls_ship_dt,sls_due_dt,sls_sales,sls_quantity,sls_price,dwh_create_date
0,SO57916,TT-M928,23024,2013-01-13,2013-01-20,2013-01-25,25.0,5,5.0,2025-04-04 20:53:39.270000+00:00
1,SO57916,TI-M602,23024,2013-01-13,2013-01-20,2013-01-25,900.0,30,1.0,2025-04-04 20:53:39.270000+00:00
2,SO57916,FE-6654,23024,2013-01-13,2013-01-20,2013-01-25,484.0,22,22.0,2025-04-04 20:53:39.270000+00:00
3,SO57916,LJ-0192-X,23024,2013-01-13,2013-01-20,2013-01-25,2500.0,50,50.0,2025-04-04 20:53:39.270000+00:00
